# ChatgaiyyaAlap MT — Day 3 (Google Colab)
Few-Shot Prompting, Both Directions

**Requires Day 1 and Day 2 done first.** This notebook pulls from your GitHub repo:
- `outputs/sampled_pairs.csv` — the same test sample used on Day 2 (do not resample)
- `outputs/fewshot_pool.csv` — the held-out pairs Day 1 set aside specifically so
  few-shot examples never overlap with the test set

**Before you start:** `Runtime → Change runtime type → T4 GPU` (optional, faster).


## Step 0 — Install dependencies

In [1]:
!pip -q install transformers accelerate sacrebleu pandas tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 2.3 MB/s eta 0:00:00


## Step 2 — Load the test sample and the few-shot candidate pool

These must be the exact files your team already agreed on from Day 1 — don't
regenerate either one here.


In [2]:
import pandas as pd
from pathlib import Path

SAMPLE_PATH = Path("/content/sampled_pairs.csv")
FEWSHOT_POOL_PATH = Path("/content/fewshot_pool.csv")

for p in (SAMPLE_PATH, FEWSHOT_POOL_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found. Run the Day 1 notebook first and make sure it pushed "
            f"outputs/ to GitHub, or manually copy the file into this repo."
        )

sample = pd.read_csv(SAMPLE_PATH)
fewshot_pool = pd.read_csv(FEWSHOT_POOL_PATH)

print(f"Test sample: {len(sample)} pairs (same as Day 2)")
print(f"Few-shot candidate pool: {len(fewshot_pool)} pairs (held out, not in test sample)")
fewshot_pool.head()


Test sample: 60 pairs (same as Day 2)
Few-shot candidate pool: 8 pairs (held out, not in test sample)


,bangla,chatgaiya,bangla_len,chatgaiya_len
0,নানি বলে গাছে বরই পারিস না,নানি কইয়্যরে গাছেত বরই ন ফাড়োস,6,6
1,আমি মাংস পছন্দ করি না,অ্যাঁই গোস্ত ফসন্দো ন গরি,5,5
2,ভারত এসে গেছি,ভারত আই গেয়্যি,3,3
3,ছেলের আশায় সে বিয়ে করল আবার,ফোয়ার আশাত ইতে বিয়া গরিল আবার,6,6
4,টানতে টানতে নিজে পুকুরে থাকে,টাইনতে টাইনতে নিজে পইরত থাহে,5,5


## Step 3 — Select 3-5 representative few-shot examples

Picking on purpose, not randomly — pull a short, a medium, and a long sentence so the
model sees range, rather than 5 examples that all look alike. Pick the same fixed
examples for both directions so B2C and C2B are actually comparable.

**Be ready to explain your choices** — the assignment explicitly asks "what made you
choose these specific few-shot examples?"


In [3]:
fewshot_pool["bangla_len"] = fewshot_pool["bangla"].str.split().str.len()

# Sort examples by Bangla sentence length
sorted_pool = fewshot_pool.sort_values("bangla_len").reset_index(drop=True)
n = len(sorted_pool)

NUM_FEWSHOT = 4

# Select examples from short, medium, and long sentences
picks_idx = sorted(set([
    0,              # shortest
    n // 3,         # around 1/3
    (2 * n) // 3,   # around 2/3
    n - 1           # longest
]))[:NUM_FEWSHOT]

fewshot_examples = sorted_pool.iloc[picks_idx][
    ["bangla", "chatgaiya", "bangla_len"]
].reset_index(drop=True)

print(f"Selected {len(fewshot_examples)} few-shot examples:")
fewshot_examples

Selected 4 few-shot examples:


,bangla,chatgaiya,bangla_len
0,ভারত এসে গেছি,ভারত আই গেয়্যি,3
1,টানতে টানতে নিজে পুকুরে থাকে,টাইনতে টাইনতে নিজে পইরত থাহে,5
2,নানি বলে গাছে বরই পারিস না,নানি কইয়্যরে গাছেত বরই ন ফাড়োস,6
3,তোমার কলেজ কয়টা থেকে কয়টা পর্যন্ত,তোঁয়ার কলেজ কয়ডা তুন কয়ডা পইযুন্ত,6


In [4]:
# Sanity check: confirm zero overlap with the test sample (must be true — few-shot
# examples leaking into the test set would invalidate the whole comparison)
overlap = set(fewshot_examples["bangla"]) & set(sample["bangla"])
assert not overlap, f"Few-shot examples overlap with test sample: {overlap}"
print("No overlap between few-shot examples and test sample. Good.")


No overlap between few-shot examples and test sample. Good.


## Step 4 — Build the few-shot prompt templates

Same instruction framing as the Day 2 zero-shot prompt, but now preceded by the
selected example pairs so the model has in-context demonstrations of the actual
mapping before it sees the real sentence.

Saved as `.txt` files under `prompts/`, matching the Day 2 convention.


In [5]:
import os
os.makedirs("prompts", exist_ok=True)

def build_fewshot_block(examples, source_key, target_key, source_label, target_label):
    lines = []
    for _, row in examples.iterrows():
        lines.append(f"{source_label}: {row[source_key]}\n{target_label}: {row[target_key]}")
    return "\n\n".join(lines)

fewshot_block_b2c = build_fewshot_block(
    fewshot_examples, "bangla", "chatgaiya", "Standard Bangla", "Chittagonian"
)
fewshot_block_c2b = build_fewshot_block(
    fewshot_examples, "chatgaiya", "bangla", "Chittagonian", "Standard Bangla"
)

few_shot_b2c_template = (
    "Translate Standard Bangla sentences into the Chittagonian (Chatgaiya) dialect "
    "spoken in southeastern Bangladesh. Chatgaiya is a distinct regional dialect, not "
    "Standard Bangla. Follow the pattern shown in these examples, then translate the "
    "final sentence. Reply with only the translated sentence, nothing else.\n\n"
    f"{fewshot_block_b2c}\n\n"
    "Standard Bangla: {text}\n"
    "Chittagonian:"
)

few_shot_c2b_template = (
    "Translate Chittagonian (Chatgaiya) dialect sentences from southeastern Bangladesh "
    "into Standard Bangla. Follow the pattern shown in these examples, then translate "
    "the final sentence. Reply with only the translated sentence, nothing else.\n\n"
    f"{fewshot_block_c2b}\n\n"
    "Chittagonian: {text}\n"
    "Standard Bangla:"
)

with open("prompts/few_shot_b2c_v1.txt", "w", encoding="utf-8") as f:
    f.write(few_shot_b2c_template)
with open("prompts/few_shot_c2b_v1.txt", "w", encoding="utf-8") as f:
    f.write(few_shot_c2b_template)

print(few_shot_b2c_template)


Translate Standard Bangla sentences into the Chittagonian (Chatgaiya) dialect spoken in southeastern Bangladesh. Chatgaiya is a distinct regional dialect, not Standard Bangla. Follow the pattern shown in these examples, then translate the final sentence. Reply with only the translated sentence, nothing else.

Standard Bangla: ভারত এসে গেছি
Chittagonian: ভারত আই গেয়্যি

Standard Bangla: টানতে টানতে নিজে পুকুরে থাকে
Chittagonian: টাইনতে টাইনতে নিজে পইরত থাহে

Standard Bangla: নানি বলে গাছে বরই পারিস না
Chittagonian: নানি কইয়্যরে গাছেত বরই ন ফাড়োস

Standard Bangla: তোমার কলেজ কয়টা থেকে কয়টা পর্যন্ত
Chittagonian: তোঁয়ার কলেজ কয়ডা তুন কয়ডা পইযুন্ত

Standard Bangla: {text}
Chittagonian:


## Step 5 — Load the model

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # keep consistent with Day 1/2 unless your team agreed to change it

print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
model.eval()
print(f"Loaded on device: {device}")

GEN_CONFIG = dict(max_new_tokens=100, temperature=0.3, do_sample=True)   # same as Day 2, keep it identical for fair comparison
print("Generation config:", GEN_CONFIG)


Loading Qwen/Qwen2.5-1.5B-Instruct ...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded on device: cuda
Generation config: {'max_new_tokens': 100, 'temperature': 0.3, 'do_sample': True}


## Step 6 — Run few-shot translation across the same sample, both directions

Same `sample` used on Day 2 — the few-shot examples are baked into the template, the
sentence being translated is substituted at `{text}`.


In [7]:
import time
from tqdm.auto import tqdm

def translate(text, template):
    prompt_text = template.format(text=text)
    messages = [{"role": "user", "content": prompt_text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    start = time.time()
    with torch.no_grad():
        output_ids = model.generate(**inputs, **GEN_CONFIG)
    latency = time.time() - start

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    text_out = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return text_out, latency


def run_direction(df, template, source_col, ref_col):
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        prediction, latency = translate(row[source_col], template)
        rows.append({
            "source": row[source_col],
            "reference": row[ref_col],
            "prediction": prediction,
            "latency_sec": round(latency, 3),
        })
    return rows


print("Running few-shot: Bangla -> Chatgaiya ...")
results_b2c = run_direction(sample, few_shot_b2c_template, source_col="bangla", ref_col="chatgaiya")

print("Running few-shot: Chatgaiya -> Bangla ...")
results_c2b = run_direction(sample, few_shot_c2b_template, source_col="chatgaiya", ref_col="bangla")


Running few-shot: Bangla -> Chatgaiya ...


  0%|          | 0/60 [00:00<?, ?it/s]

Running few-shot: Chatgaiya -> Bangla ...


  0%|          | 0/60 [00:00<?, ?it/s]

## Step 7 — Score each example with BLEU and chrF

In [8]:
import sacrebleu, statistics

def score_pair(prediction, reference):
    if not prediction.strip():
        return 0.0, 0.0
    bleu = sacrebleu.sentence_bleu(prediction, [reference]).score / 100
    chrf = sacrebleu.sentence_chrf(prediction, [reference]).score / 100
    return round(bleu, 4), round(chrf, 4)


def add_scores(rows):
    for r in rows:
        bleu, chrf = score_pair(r["prediction"], r["reference"])
        r["bleu"] = bleu
        r["chrf"] = chrf
    return rows


results_b2c = add_scores(results_b2c)
results_c2b = add_scores(results_c2b)

def avg(rows, key):
    return round(statistics.mean(r[key] for r in rows), 4)

print("Bangla -> Chatgaiya  avg BLEU:", avg(results_b2c, "bleu"), " avg chrF:", avg(results_b2c, "chrf"))
print("Chatgaiya -> Bangla  avg BLEU:", avg(results_c2b, "bleu"), " avg chrF:", avg(results_c2b, "chrf"))


Bangla -> Chatgaiya  avg BLEU: 0.0646  avg chrF: 0.2205
Chatgaiya -> Bangla  avg BLEU: 0.097  avg chrF: 0.2578


## Step 8 — Compare against Day 2 zero-shot

This is the actual point of Day 3 — did few-shot help, hurt, or do nothing? Pull
Day 2's numbers back in and compare directly, per direction.


In [9]:
import json

with open("/content/day2_zeroshot_b2c.json", encoding="utf-8") as f:
    day2_b2c = json.load(f)
with open("/content/day2_zeroshot_c2b.json", encoding="utf-8") as f:
    day2_c2b = json.load(f)

print(f"{'Direction':<20}{'Technique':<12}{'avg BLEU':<10}{'avg chrF':<10}")
print(f"{'B2C':<20}{'zero-shot':<12}{day2_b2c['metrics']['avg_bleu']:<10}{day2_b2c['metrics']['avg_chrf']:<10}")
print(f"{'B2C':<20}{'few-shot':<12}{avg(results_b2c,'bleu'):<10}{avg(results_b2c,'chrf'):<10}")
print(f"{'C2B':<20}{'zero-shot':<12}{day2_c2b['metrics']['avg_bleu']:<10}{day2_c2b['metrics']['avg_chrf']:<10}")
print(f"{'C2B':<20}{'few-shot':<12}{avg(results_c2b,'bleu'):<10}{avg(results_c2b,'chrf'):<10}")
print()
print("If few-shot barely moved the needle or made things worse, that's a real finding")
print("— don't just assume you did something wrong. Note *why* you think that happened")
print("(e.g. examples too different in style/length from the failing test sentences,")
print("model too small to exploit in-context examples, dialect too unfamiliar to it).")


Direction           Technique   avg BLEU  avg chrF  
B2C                 zero-shot   0.0415    0.1389    
B2C                 few-shot    0.0646    0.2205    
C2B                 zero-shot   0.0469    0.1598    
C2B                 few-shot    0.097     0.2578    

If few-shot barely moved the needle or made things worse, that's a real finding
— don't just assume you did something wrong. Note *why* you think that happened
(e.g. examples too different in style/length from the failing test sentences,
model too small to exploit in-context examples, dialect too unfamiliar to it).


## Step 9 — Spot-check worst-scoring examples and update the issue log


In [10]:
worst = sorted(results_b2c + results_c2b, key=lambda r: r["chrf"])[:5]
for r in worst:
    print(f"SRC:  {r['source']}")
    print(f"REF:  {r['reference']}")
    print(f"PRED: {r['prediction']}")
    print(f"chrF: {r['chrf']}")
    print("-" * 60)


SRC:  ন্যাকামো করা না
REF:  আজল হোয়লদ্দে না
PRED: নায়কামো করা না
chrF: 0.0524
------------------------------------------------------------
SRC:  হক্কল হথা কি হইয়্যুম অ্যাঁই
REF:  সব কথা কি বলবো আমি
PRED: হক্কল হতে কি হয়েছিল অনুই
chrF: 0.0657
------------------------------------------------------------
SRC:  বউকে ধরে ঘুষি দিতে হবে
REF:  বউরে ধরি কিলদন অইব্যু
PRED: ভারত আই গেয়্যি
chrF: 0.0692
------------------------------------------------------------
SRC:  সবাইকে অনুরোধ করি আমি
REF:  অক্কলরে অনুরোধ গরি অ্যাঁই
PRED: সবাইকে আইন করি আমি
chrF: 0.0732
------------------------------------------------------------
SRC:  অ্যাঁই নিজর ডাইল ভাত শেষ গইজ্জি
REF:  আমি নিজের ডাল ভাত শেষ করেছি
PRED: অনুভব নেই মনে রাখি
chrF: 0.0732
------------------------------------------------------------


In [12]:
issue_notes = [
     "b2c few-shot: model copied phrasing from example 2 verbatim into an unrelated sentence",
     "few-shot didn't fix c2b defaulting-to-Bangla issue seen in Day 2",
]

os.makedirs("logs", exist_ok=True)
with open("logs/issue_log.txt", "a", encoding="utf-8") as f:
    f.write(f"\n--- Day 3 few-shot ({pd.Timestamp.utcnow()}) ---\n")
    f.write(f"Few-shot examples used ({len(fewshot_examples)}): {fewshot_examples['bangla'].tolist()}\n")
    if issue_notes:
        for note in issue_notes:
            f.write(f"- {note}\n")
    else:
        f.write("- (fill in real observations before submitting — don't leave this empty)\n")

print("Logged to logs/issue_log.txt — go edit issue_notes above with your real findings.")


Logged to logs/issue_log.txt — go edit issue_notes above with your real findings.


## Step 10 — Save as `day3_fewshot_b2c.json` and `day3_fewshot_c2b.json`

In [11]:
from datetime import datetime, timezone

def build_output(direction, template_file, results):
    return {
        "dataset": "ChatgaiyyaAlap",
        "direction": direction,
        "technique": "few_shot",
        "model": MODEL_NAME,
        "prompt_template": template_file,
        "num_fewshot_examples": len(fewshot_examples),
        "fewshot_examples_used": fewshot_examples.to_dict(orient="records"),
        "generation_config": GEN_CONFIG,
        "num_examples": len(results),
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "results": results,
        "metrics": {
            "avg_bleu": avg(results, "bleu"),
            "avg_chrf": avg(results, "chrf"),
            "avg_latency_sec": round(statistics.mean(r["latency_sec"] for r in results), 3),
        },
    }

output_b2c = build_output("bangla_to_chatgaiya", "prompts/few_shot_b2c_v1.txt", results_b2c)
output_c2b = build_output("chatgaiya_to_bangla", "prompts/few_shot_c2b_v1.txt", results_c2b)

os.makedirs("outputs", exist_ok=True)
with open("outputs/day3_fewshot_b2c.json", "w", encoding="utf-8") as f:
    json.dump(output_b2c, f, ensure_ascii=False, indent=2)
with open("outputs/day3_fewshot_c2b.json", "w", encoding="utf-8") as f:
    json.dump(output_c2b, f, ensure_ascii=False, indent=2)

print("Saved outputs/day3_fewshot_b2c.json")
print("Saved outputs/day3_fewshot_c2b.json")


Saved outputs/day3_fewshot_b2c.json
Saved outputs/day3_fewshot_c2b.json


---
**Recap — Day 3 checklist:**
1. ✅ Selected 3-5 representative few-shot examples, excluded from the test sample
2. ✅ Built few-shot prompt templates for both directions
3. ✅ Ran both across the same Day 2 sample
4. ✅ Stored raw outputs alongside references
5. ✅ Saved `day3_fewshot_b2c.json` and `day3_fewshot_c2b.json`

**Be ready to answer:**
- Why did you choose these specific few-shot examples?
- Did few-shot actually improve scores over Day 2 zero-shot, in both directions equally?
- Any signs of the model copying example phrasing verbatim instead of generalizing?

Next: Day 4 — dictionary-augmented prompt variant.
